In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Cargar los datos
csv_path = r"C:\Users\japje\Documents\B.G\wiki_hybrid_chunks_300.csv"
df = pd.read_csv(csv_path)
texts = df["chunk_text"].tolist()
id_to_text = df[["chunk_id", "chunk_text"]].reset_index(drop=True)

# Embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
# OPTIMIZACIÓN: Procesar en lotes grandes y sin mostrar barra de progreso para mayor velocidad
embeddings = embed_model.encode(
    texts, convert_to_numpy=True, batch_size=128, show_progress_bar=False
)
dim = embeddings.shape[1]

# FAISS index
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

# TinyLLaMA fallback
tinyllama_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(tinyllama_model_id)
model = AutoModelForCausalLM.from_pretrained(tinyllama_model_id)
local_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=-1)

# OpenAI setup
openai_api_key = "OPENAI_API_KEY"  # Reemplaza con tu clave real
client = OpenAI(api_key=openai_api_key)

def ask_question(query, k=5, context_chunks=3):
    """
    Responde una pregunta usando RAG con OpenAI y TinyLLaMA como fallback.
    Args:
        query (str): Pregunta del usuario.
        k (int): Número de chunks a recuperar.
        context_chunks (int): Número de chunks a usar como contexto.
    Returns:
        dict: {'answer': str, 'chunks': list, 'model': str}
    """
    # OPTIMIZACIÓN: Solo usar los chunks necesarios para el contexto
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, k)
    retrieved_chunks = [id_to_text.iloc[idx]["chunk_text"] for idx in indices[0]]
    context = "\n".join(retrieved_chunks[:context_chunks])
    max_context_len = 1500
    context = context[:max_context_len]
    prompt = f"""You are a helpful assistant. Use the following context to answer the question:

Context:
{context}

Question: {query}
Answer:"""

    answer = None
    model_used = None

    try:
        # OPTIMIZACIÓN: Reducir temperatura para respuestas más consistentes
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=400
        )
        answer = response.choices[0].message.content
        model_used = "OpenAI GPT-3.5"
    except Exception as e:
        print(f"⚠️ OpenAI failed: {e}")
        # OPTIMIZACIÓN: Limitar tokens generados para mayor velocidad
        output = local_pipe(prompt, max_new_tokens=200, temperature=0.5, do_sample=True)
        answer = output[0]["generated_text"].split("Answer:")[-1].strip()
        model_used = "TinyLLaMA"

    return {
        "answer": answer,
        "chunks": retrieved_chunks,
        "model": model_used
    }

# OPCIÓN MÁS OPTIMIZADA: Uso de memoria y procesamiento eficiente
def ask_question_optimized(query, k=3):
    """
    Versión optimizada para consultas rápidas y bajo consumo de recursos.
    - Usa menos chunks y menos tokens.
    - Reduce el tamaño del contexto.
    - Ajusta parámetros para respuestas más rápidas.
    """
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, k)
    retrieved_chunks = [id_to_text.iloc[idx]["chunk_text"] for idx in indices[0]]
    # Solo el primer chunk como contexto para máxima velocidad
    context = retrieved_chunks[0][:800]
    prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"

    answer = None
    model_used = None

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=200
        )
        answer = response.choices[0].message.content
        model_used = "OpenAI GPT-3.5 (Optimized)"
    except Exception as e:
        print(f"⚠️ OpenAI failed: {e}")
        output = local_pipe(prompt, max_new_tokens=100, temperature=0.3, do_sample=True)
        answer = output[0]["generated_text"].split("Answer:")[-1].strip()
        model_used = "TinyLLaMA (Optimized)"

    return {
        "answer": answer,
        "chunks": retrieved_chunks,
        "model": model_used
    }

# Ejemplo de uso:
if __name__ == "__main__":
    print("=== Respuesta estándar ===")
    result = ask_question("who is bob brown", k=5)
    print(f"\nModelo: {result['model']}\nRespuesta:\n{result['answer']}\n")

    print("=== Respuesta optimizada ===")
    result_opt = ask_question_optimized("who is bob brown", k=3)
    print(f"\nModelo: {result_opt['model']}\nRespuesta:\n{result_opt['answer']}\n")